In [15]:
import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.iolib.summary2 import summary_col
import statsmodels.api as sm
import statsmodels.formula.api as smf

这个是使用起源国+目的国+时间固定

读取数据并调整顺序

In [16]:
# 读取数据
model_data_path = '../data/processed/merged_dataset_cleaned.csv'
model_df = pd.read_csv(model_data_path)

def add_interaction_terms(df, spei_lag_var):
    """
    添加 SPEI 滞后变量的交互项。
    
    Parameters:
    df: pd.DataFrame
        数据框
    spei_lag_var: str
        SPEI 滞后变量的列名
    
    Returns:
    pd.DataFrame
        添加交互项后的数据框
    """
    df['spei_ma_interact'] = df[spei_lag_var] * df['o_MA']
    df['spei_bf_interact'] = df[spei_lag_var] * df['border_friction_ij']
    return df

model_df = add_interaction_terms(model_df, 'spei_lag1')



In [17]:
# 按照指定顺序调整列顺序并删除
ordered_columns = [
    # ISO 列
    'origin_iso3', 'destination_iso3',
    # 日期相关
    'migration_date', 'year', 'month',
    # 因变量
    'flow', 'log_flow',
    # 重点自变量
    'o_spei', 'spei_lag1', 'spei_lag2', 'spei_lag3', 'o_MA', 'border_friction_ij',
    # 交互项
    'spei_ma_interact', 'spei_bf_interact',
    # 控制变量
    'o_gdp_pc', 'd_gdp_pc', 'o_pop', 'd_pop', 'o_urban', 'd_urban',
    'o_unemp', 'd_unemp', 'o_political_stability', 'd_political_stability',
    'control_distwces'
]

# 重新排列列顺序
model_df = model_df[ordered_columns]

In [18]:
float_cols = model_df.select_dtypes(include=['float64']).columns
model_df[float_cols] = model_df[float_cols].astype('float32')

#### 1. 数据质量检查

##### 时间变量处理
生成 唯一的时间 ID（方便做时间固定效应）：

In [19]:
# 创建 time_id，从 2019 年开始的连续数字
model_df['time_id'] = (
    (pd.to_datetime(model_df['migration_date']).dt.year - 2019) * 12 +
    pd.to_datetime(model_df['migration_date']).dt.month
)

In [20]:
model_df 

,origin_iso3,destination_iso3,migration_date,year,month,flow,log_flow,o_spei,spei_lag1,spei_lag2,...,o_pop,d_pop,o_urban,d_urban,o_unemp,d_unemp,o_political_stability,d_political_stability,control_distwces,time_id
0,ARE,AND,2019-01-01,2019,1,0,0.000000,0.013769,0.345254,-0.196752,...,16.061079,11.244706,86.789001,87.984001,2.236,5.542,0.667538,-0.091492,8.563919,1
1,ARE,AND,2019-02-01,2019,2,12,2.564949,-0.017917,0.013769,0.345254,...,16.061079,11.244706,86.789001,87.984001,2.236,5.542,0.667538,-0.091492,8.563919,2
2,ARE,AND,2019-03-01,2019,3,0,0.000000,0.490999,-0.017917,0.013769,...,16.061079,11.244706,86.789001,87.984001,2.236,5.542,0.667538,-0.091492,8.563919,3
3,ARE,AND,2019-04-01,2019,4,1,0.693147,0.797000,0.490999,-0.017917,...,16.061079,11.244706,86.789001,87.984001,2.236,5.542,0.667538,-0.091492,8.563919,4
4,ARE,AND,2019-05-01,2019,5,3,1.386294,0.580897,0.797000,0.490999,...,16.061079,11.244706,86.789001,87.984001,2.236,5.542,0.667538,-0.091492,8.563919,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1319781,ZWE,ZMB,2022-08-01,2022,8,138,4.934474,0.326137,0.494725,0.959915,...,16.592405,16.818861,32.395000,45.761002,9.540,5.199,-0.894974,0.161332,6.263538,44
1319782,ZWE,ZMB,2022-09-01,2022,9,162,5.093750,-0.502948,0.326137,0.494725,...,16.592405,16.818861,32.395000,45.761002,9.540,5.199,-0.894974,0.161332,6.263538,45
1319783,ZWE,ZMB,2022-10-01,2022,10,149,5.010635,-0.593602,-0.502948,0.326137,...,16.592405,16.818861,32.395000,45.761002,9.540,5.199,-0.894974,0.161332,6.263538,46
1319784,ZWE,ZMB,2022-11-01,2022,11,104,4.653960,-0.331173,-0.593602,-0.502948,...,16.592405,16.818861,32.395000,45.761002,9.540,5.199,-0.894974,0.161332,6.263538,47


In [21]:
df = model_df.reset_index(drop=False).copy()

In [22]:
if 'pair_id' not in df.columns:
    df['pair_id'] = df['origin_iso3'].astype(str) + '_' + df['destination_iso3'].astype(str)

##### 3. 双边 ID 创建
固定效应要区分国家对：

In [23]:
absorbs = df[['origin_iso3','destination_iso3','time_id']]

In [24]:
controls = [
    'o_gdp_pc', 'd_gdp_pc',
    'o_pop', 'd_pop',
    'o_urban', 'd_urban',
    'o_unemp', 'd_unemp',
    'o_political_stability', 'd_political_stability',
    'control_distwces'
]

In [25]:
# def fit_ols_absorb(formula, data):
#     # 丢弃缺失
#     use_cols = set(['log_flow']) | set(controls) | {'spei_lag1','o_MA','border_friction_ij',
#                                                     'spei_lag1:o_MA','spei_lag1:border_friction_ij'}
#     use_cols = [c for c in use_cols if c in data.columns]
#     used = data.dropna(subset=[c for c in use_cols]).copy()

#     res = smf.ols(formula=formula, data=used).fit(
#         cov_type='cluster',
#         cov_kwds={'groups': used['pair_id']},   # 按 pair 聚类标准误
#         absorb = absorbs.loc[used.index]        # 一次吸收 origin+destination+time 三类 FE
#     )
#     return res

def fit_ols_absorb(formula, data):
    # 丢缺失
    used = data.copy()
    used = used.replace([np.inf, -np.inf], np.nan).dropna(subset=['log_flow','spei_lag1'] + controls)
    # 拟合（按 pair 聚类；吸收三类 FE）
    res = smf.ols(formula=formula, data=used).fit(
        cov_type='cluster',
        cov_kwds={'groups': used['pair_id']},
        absorb=absorbs.loc[used.index]
    )
    return res

#### 3. 模型搭建

In [26]:
# Baseline
f_baseline = "log_flow ~ spei_lag1 + " + " + ".join(controls)

# Model 2：+ o_MA 与交互
f_m2 = "log_flow ~ spei_lag1 + o_MA + spei_ma_interact + " + " + ".join(controls)

# Model 3：+ border_friction_ij 与交互
f_m3 = "log_flow ~ spei_lag1 + border_friction_ij + spei_bf_interact + " + " + ".join(controls)

# Model 4：联合
f_joint = ("log_flow ~ spei_lag1 + o_MA + border_friction_ij + "
           "spei_ma_interact + spei_bf_interact + "
           + " + ".join(controls))

##### 模型定义

#### 不同的归回模型设定

双边固定+时间固定

In [27]:
res_baseline = fit_ols_absorb(f_baseline, df)
res_m2       = fit_ols_absorb(f_m2,       df)
res_m3       = fit_ols_absorb(f_m3,       df)
res_joint    = fit_ols_absorb(f_joint,    df)

##### 回归分析

In [28]:
print("\n=== Baseline: O+D FE + Time FE（statsmodels 吸收） ===")
print(res_baseline.summary())
print("\n=== Model 2: spei × o_MA | O+D FE + Time FE ===")
print(res_m2.summary())
print("\n=== Model 3: spei × border_friction | O+D FE + Time FE ===")
print(res_m3.summary())
print("\n=== Joint: both interactions | O+D FE + Time FE ===")
print(res_joint.summary())


=== Baseline: O+D FE + Time FE（statsmodels 吸收） ===
                            OLS Regression Results                            
Dep. Variable:               log_flow   R-squared:                       0.328
Model:                            OLS   Adj. R-squared:                  0.328
Method:                 Least Squares   F-statistic:                     1101.
Date:                Wed, 13 Aug 2025   Prob (F-statistic):               0.00
Time:                        22:02:31   Log-Likelihood:            -2.3791e+06
No. Observations:             1319786   AIC:                         4.758e+06
Df Residuals:                 1319773   BIC:                         4.758e+06
Df Model:                          12                                         
Covariance Type:              cluster                                         
                            coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------

g:\python\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 14, but rank is 12
  warnings.warn('covariance of constraints does not have full '


                            OLS Regression Results                            
Dep. Variable:               log_flow   R-squared:                       0.333
Model:                            OLS   Adj. R-squared:                  0.333
Method:                 Least Squares   F-statistic:                     1090.
Date:                Wed, 13 Aug 2025   Prob (F-statistic):               0.00
Time:                        22:02:32   Log-Likelihood:            -2.3733e+06
No. Observations:             1319786   AIC:                         4.747e+06
Df Residuals:                 1319771   BIC:                         4.747e+06
Df Model:                          14                                         
Covariance Type:              cluster                                         
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                -6.46

g:\python\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 16, but rank is 14
  warnings.warn('covariance of constraints does not have full '
